In [ ]:
import torch
import math


class LinearLayer:
    def __init__(self,in_features,out_features,bias=False):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.has_bias= bias
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias = None

    def forward(self,x):
        self.x  = x
        out = self.x @ self.weights.T
        if self.has_bias:
            out = out+self.bias
        
        return out


    def backward(self,grad_out):
        x_shape = self.x.shape
        x_flat = self.x.flatten(0,-2)
        grad_flat = grad_out.flatten(0,-2)

        grad_inputs = grad_flat @ self.weights
        self.weights.grad = grad_flat.T @ x_flat
        if self.has_bias:
            self.bias.grad = grad_flat.sum(dim=0)

        grad_inputs = grad_inputs.reshape(x_shape)
        return grad_inputs

class softmax:

    def forward(self,scores):
        max_score = torch.max(scores,dim=-1,keepdim=True).values
        scores_exp = torch.exp(scores-max_score)
        self.scores_sum = scores_exp.sum(dim=-1,keepdim=True)
        self.out = scores_exp/self.scores_sum
        return self.out

    
    def backward(self,grad_attn):

        sum_term = (grad_attn *self.out ).sum(dim=-1, keepdim=True)
        grad_scores = self.out * (grad_out - sum_term)

        return grad_scores
         

class CausalSelfAttention:
    def __init__(self,num_dims,num_heads):
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims,num_dims,bias=False)
        self.w_k = LinearLayer(num_dims,num_dims,bias=False)
        self.w_v = LinearLayer(num_dims,num_dims,bias=False)

        self.proj_out = LinearLayer(num_dims,num_dims,bias=True)
        self.softmax = softmax()

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        self.Q = Q
        self.K = K
        self.V = V
        
        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool,device=scores.device),diagonal=1)

        scores = scores.masked_fill(masks,float("-inf"))

        self.attn_scores = self.softmax.forward(scores)
        self.out = self.attn_scores @ self.V
        self.out = self.out.transpose(1,2).contiguous().view(B,T,D)
        self.out = self.proj_out(self.out)

        return self.out

    def backward(self,grad_out):

        grad_proj = self.proj_out.backward(grad_out)
        grad_proj = grad_proj.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        
        grad_attn = grad_proj @ self.V.transpose(-2,-1)
        grad_V = self.attn_scores.transpose(-2,-1) @ grad_proj

        grad_scores = self.softmax.backward(grad_attn)   

        scale = 1.0 / math.sqrt(self.head_dims)
        grad_scores = grad_scores * scale
        
        grad_q = grad_scores @ self.K
        grad_k = grad_scores.transpose(-2,-1) @ self.Q
        

        grad_Q = grad_Q.transpose(1, 2).contiguous().view(B, T, D)
        grad_K = grad_K.transpose(1, 2).contiguous().view(B, T, D)
        grad_V = grad_V.transpose(1, 2).contiguous().view(B, T, D)

        grad_x_q = self.w_q.backward(grad_Q)
        grad_x_k = self.w_k.backward(grad_K)
        grad_x_v = self.w_v.backward(grad_V)

        grad_x = grad_x_q+grad_x_k+grad_x_v

        return grad_x

    def parameters(self):
        return [self.w_q.weights, self.w_k.weights, self.w_v.weights, self.proj_out.weights, self.proj_out.bias]
        

        
    

        




grad_proj ->shape(B,H,T,d_k)
attn_scores ->shape(B,H,T,T)
V ->shape(B,H,T,d_k)
we can assume this as the ,cols of the attn(T) is multiplied with the rows of the V(T)
so the calcs are (i,j) @ (j,k) -> (i,k) as the k is the d_k
and we got the proj shape as the (B,T,i,k)
we need the grad_attn right,it is the (B,T,i,j)

so i was going in this way right
And thwn how can i actually know what to take for what?
i mean like the 
yeah when there are two  variables and then they need the derivation and then one becomes constant right so that
we can leave a place for the grad_out and then we need tp check
for which,whether it is the attn or V
this is cheat sheet 😂️




we need to find the grad_scores ->
dL/dscores = dL/attn * dattn/dscores

core : we use the grad_attention to updatee the errors from the scores
shape of the attention -> B H T T 
so ignore the B H and focus on the T T 
so it is a 2x2 matrix,we can take them as the
i,j as the i denotes the row and then j denotes the column
what exactly does the softmax does ??
it make the each column_exp value sum to the 1
and now there are two loops right i and j
now we need to sum the values 
e^j/sum(e^j) to the all j values inside that row i
now this is good and then we are gonna use the softmax formulas right
$e^{s_{i, j}}$ and then below is sum right broo
now how do we change the softmax values is the core question
so the
dL/d scores = dL/attn * d attn/scores
and 
now we introduce the two variables called the 
s and prob
s0 -> exponentiated scores
p0->probability of that exponentiated scores

ignore everything
 now we introduce the two variables called the 
        s and prob
        s-> exponentiated scores
        p->probability of that exponentiated scores

attn_scores = softmax(scores)
coming backward we knew the grad_attn and we need to find the grad_scores with the total loss
and for that
d Loss/dscores =d loss/d attn * d attn/d scores
so now we need to updatee the scores 
as the attention have the shape of the i,j we have talked earlier and then
what exactly does the attention out have ??
we have the 

    0    1   2   3    
0   p0  p1  p2  p3
1   p0  p1  p2  p3
2   p0  p1  p2  p3
3   p0  p1  p2  p3

so this is what we have
and we got the entire gradient attention ?so what it measn we have the entire loss sum
so our final eqn consists of the sum
and then now we need to find the
changes in the p0 right?
so for the p0 we need to find what does the p0 even have??
p0 ->s0/s0+s1+s2
and we can't just change via the e^0 and leave it,we need for the all the changes from the all the e^ values 
so this is where the formula comes is 
p0/s0 and p0/s1 and p0/s2
these all blend comes in and then we will get the gradient t change that single p0??
        

$$\text{grad\_attn} = \left[ \frac{\partial \mathcal{L}}{\partial p_0}, \frac{\partial \mathcal{L}}{\partial p_1}, \frac{\partial \mathcal{L}}{\partial p_2}, \dots \right]$$

so we get the grad in a way that they are the dL/d probs
so we need to find the
dL/ds0 =?
it is the dL/dp * dp/ds
and now we need to know the
dp0/ds0+ dp1/ds0+dp2/ds0

wait a min
that s0 exist in the entire row right
so that,we are gonna find this one
p0 is changed by what ?and then by which elements 
they are changed by the entire s0 s1 s2 right
and with that only we get the dL/dp0 yes ornot and it is yes

dL/dp0 ->we got from the attention from the next layers 
and we need to update what?? we need to updatee the scores,not probabilitys
and then for the updatee of the scores we do the 

how much does each scores did make the fault to make the loss so we can updatee them
and then what do the scores effect?
dp0/ds0+dp1/ds0+dp2/ds0

dl/ds0 = dl/dprobs* dprobs/d scores
we already knew the dL/dprobs this is where we get the grad_attn 
and we need to find that d _probs/d scores
so this is the (dp0/ds0+dp1/ds0+dp2/ds0)


$$\frac{\partial \mathcal{L}}{\partial s_0} = \left( \frac{\partial \mathcal{L}}{\partial p_0} \cdot \frac{\partial p_0}{\partial s_0} \right) + \left( \frac{\partial \mathcal{L}}{\partial p_1} \cdot \frac{\partial p_1}{\partial s_0} \right) + \left( \frac{\partial \mathcal{L}}{\partial p_2} \cdot \frac{\partial p_2}{\partial s_0} \right)$$


For the matching position ($p_0$ and $s_0$): $\frac{\partial p_0}{\partial s_0} = p_0(1 - p_0)$

For the neighbor positions: $\frac{\partial p_1}{\partial s_0} = -p_1 p_0$ and $\frac{\partial p_2}{\partial s_0} = -p_2 p_0$



Plugging those in gives:

$$\frac{\partial \mathcal{L}}{\partial s_0} = \frac{\partial \mathcal{L}}{\partial p_0} \cdot p_0(1 - p_0) + \frac{\partial \mathcal{L}}{\partial p_1} \cdot (-p_1 p_0) + \frac{\partial \mathcal{L}}{\partial p_2} \cdot (-p_2 p_0)$$

Notice that $p_0$ appears in every single term. If you factor $p_0$ outside:

$$\frac{\partial \mathcal{L}}{\partial s_0} = p_0 \left[ \frac{\partial \mathcal{L}}{\partial p_0} - \left( \frac{\partial \mathcal{L}}{\partial p_0} p_0 + \frac{\partial \mathcal{L}}{\partial p_1} p_1 + \frac{\partial \mathcal{L}}{\partial p_2} p_2 \right) \right]$$



Look at what that factored equation reveals:

$$\frac{\partial \mathcal{L}}{\partial s_0} = p_0 \left[ \frac{\partial \mathcal{L}}{\partial p_0} - \underbrace{\left( \frac{\partial \mathcal{L}}{\partial p_0} p_0 + \frac{\partial \mathcal{L}}{\partial p_1} p_1 + \frac{\partial \mathcal{L}}{\partial p_2} p_2 \right)}_{\text{row sum: } \sum (\text{grad\_out} \cdot p)} \right]$$

Now look at what happens if you write the exact same thing for $s_1$ and $s_2$:

$$\frac{\partial \mathcal{L}}{\partial s_1} = p_1 \left[ \frac{\partial \mathcal{L}}{\partial p_1} - \sum (\text{grad\_out} \cdot p) \right]$$

$$\frac{\partial \mathcal{L}}{\partial s_2} = p_2 \left[ \frac{\partial \mathcal{L}}{\partial p_2} - \sum (\text{grad\_out} \cdot p) \right]$$

Notice the pattern:

The inner sum $\sum (\text{grad\_out} \cdot p)$ is identical for every single score in that row. It is a single scalar number for that entire row.
For each score, you just take its own incoming gradient, subtract that shared sum, and scale by its own probability.

### The Final Vectorized Form

Because this pattern holds across all scores at the same time:

$$\text{grad\_scores} = p \odot \left( \text{grad\_out} - \sum (\text{grad\_out} \odot p) \right)$$



for the grad of the q and k
we need to know,there is a filler space which is the grad_out
so grad_q = grad_out and k relation
grad_q -> shape (B,H,T,d_k)
grad_out-> shape(B,H,T,T)
grad_k -> shape(B,H,T,d_k)
grad_q = grad_out @ self.K

